# AE4: Métricas y visualización de Bike Sharing

**Objetivo.** Recalcular con un paquete reutilizable (`analitica_bike`) las 11 tablas de resultados de AE3 (preguntas P1 a P6), comprobar que coinciden con los CSV de AE3 y verificar que el dashboard (`DashboardBike`, la app de Streamlit de `AE4/app/`) reproduce exactamente esas cifras y construye todas sus figuras.

**Por qué importa.** Las cifras del notebook, de la app y del documento deben ser idénticas. Por eso las métricas viven en `src/analitica_bike/metricas.py` y la app llama a las mismas funciones.

**Qué se mide.** Alquileres (demanda). El dataset no tiene precios ni costos: no se habla de ingresos ni de rentabilidad.

**Período analizado:** 2011-2012 (17.379 horas observadas). Fuente: UCI Bike Sharing Dataset.

| Pregunta | Tema |
|---|---|
| P1 | Horas y franjas donde se concentra la demanda |
| P2 | Demanda por tipo de día (laboral, fin de semana, festivo) y por festivo |
| P3 | Demanda por estación y por mes |
| P4 | Demanda por situación climática y por rango de temperatura |
| P5 | Participación de usuarios casuales y registrados |
| P6 | Variación de la demanda entre 2011 y 2012 |

## 1. Preparación del entorno y carga de datos

In [1]:
import sys
import warnings
from pathlib import Path

import pandas as pd
from pandas.testing import assert_frame_equal

warnings.filterwarnings("ignore", message="IProgress not found")

# Busca la raíz del repositorio subiendo desde el directorio actual (sin rutas absolutas)
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "analitica_bike").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from analitica_bike import DashboardBike, datos, metricas

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

df = datos.cargar_dataset()
print(f"Dataset preparado: {len(df):,} filas x {df.shape[1]} columnas".replace(",", "."))
print(f"Período: {df['fecha'].min():%Y-%m-%d} a {df['fecha'].max():%Y-%m-%d}")
df.head()

Dataset preparado: 17.379 filas x 23 columnas
Período: 2011-01-01 a 2012-12-31


,instant,fecha,fecha_hora,anio,mes,nombre_mes,dia_semana,hora,franja_horaria,tipo_dia,nombre_festivo,tipo_festivo,estacion,clima,temp_c,sensacion_c,rango_temperatura,humedad_pct,viento_original,usuarios_casuales,usuarios_registrados,alquileres_total,pct_casual
0,1,2011-01-01,2011-01-01 00:00:00,2011,1,Enero,Sábado,0,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,3.28,3.0,Frío (< 10 °C),81.0,0.0,3,13,16,18.75
1,2,2011-01-01,2011-01-01 01:00:00,2011,1,Enero,Sábado,1,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,2.34,2.0,Frío (< 10 °C),80.0,0.0,8,32,40,20.00
2,3,2011-01-01,2011-01-01 02:00:00,2011,1,Enero,Sábado,2,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,2.34,2.0,Frío (< 10 °C),80.0,0.0,5,27,32,15.62
3,4,2011-01-01,2011-01-01 03:00:00,2011,1,Enero,Sábado,3,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,3.28,3.0,Frío (< 10 °C),75.0,0.0,3,10,13,23.08
4,5,2011-01-01,2011-01-01 04:00:00,2011,1,Enero,Sábado,4,Madrugada (0-5 h),Fin de semana,No festivo,No festivo,Invierno,Despejado o pocas nubes,3.28,3.0,Frío (< 10 °C),75.0,0.0,0,1,1,0.00


## 2. Recalcular cada tabla y compararla con AE3

Cada función de `analitica_bike.metricas` recibe el dataset preparado y devuelve la tabla equivalente. Se compara con `pandas.testing.assert_frame_equal` contra el CSV que exportó AE3 (tolerancia relativa de `1e-6` en decimales y sin exigir que el entero sea `int64` o `int32`). Si algo difiere, el notebook se detiene con el detalle de la diferencia.

In [2]:
TABLAS = {}
for nombre, funcion in metricas.FUNCIONES.items():
    calculada = funcion(df)
    esperada = datos.cargar_tabla(nombre)
    assert_frame_equal(calculada, esperada, check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-9)
    TABLAS[nombre] = calculada
    print(f"coincide: {nombre} ({len(calculada)} filas) es idéntica a la tabla de AE3")

print(f"\nLas {len(TABLAS)} tablas recalculadas coinciden con las de AE3.")

coincide: P1_demanda_por_franja_horaria (5 filas) es idéntica a la tabla de AE3
coincide: P1_demanda_por_hora_del_dia (24 filas) es idéntica a la tabla de AE3
coincide: P2_demanda_por_tipo_de_dia (3 filas) es idéntica a la tabla de AE3
coincide: P2_demanda_por_tipo_de_dia_y_franja (15 filas) es idéntica a la tabla de AE3
coincide: P2_demanda_por_festivo (11 filas) es idéntica a la tabla de AE3
coincide: P3_demanda_por_estacion (4 filas) es idéntica a la tabla de AE3
coincide: P3_demanda_por_mes (12 filas) es idéntica a la tabla de AE3
coincide: P4_demanda_por_situacion_climatica (4 filas) es idéntica a la tabla de AE3
coincide: P4_demanda_por_rango_de_temperatura (4 filas) es idéntica a la tabla de AE3
coincide: P5_participacion_casuales_y_registrados (16 filas) es idéntica a la tabla de AE3
coincide: P6_variacion_2011_2012 (3 filas) es idéntica a la tabla de AE3

Las 11 tablas recalculadas coinciden con las de AE3.


## 3. El dashboard sin filtros reproduce las cifras de AE3

`DashboardBike` es la clase que usa la app de Streamlit. Sin filtros, sus indicadores y sus tablas deben ser exactamente los de AE3 (que ya se verificaron arriba contra los CSV). Se comprueban los totales, las horas y las cifras principales de cada pregunta.

In [3]:
dash = DashboardBike.desde_csv()
k = dash.kpis()

assert k["Horas observadas"] == len(df)
assert k["Alquileres totales"] == int(df["alquileres_total"].sum())
assert k["% del dataset completo"] == 100
assert not dash.filtros_activos()
print({nombre: round(float(v), 2) for nombre, v in k.items()})

# Las tablas que usa el dashboard son las de AE3
for nombre in metricas.FUNCIONES:
    assert_frame_equal(dash.tabla(nombre), datos.cargar_tabla(nombre), check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-9)
assert_frame_equal(dash.tabla_p6(), datos.cargar_tabla("P6_variacion_2011_2012"), check_dtype=False, check_exact=False, rtol=1e-6, atol=1e-9)
print("Las 11 tablas del dashboard (sin filtros) coinciden con AE3.")

{'Horas observadas': 17379.0, 'Alquileres totales': 3292679.0, 'Promedio por hora': 189.46, '% de casuales': 18.83, '% del dataset completo': 100.0}


Las 11 tablas del dashboard (sin filtros) coinciden con AE3.


In [4]:
# Cifras principales que el dashboard escribe en sus textos, contra las tablas de AE3
ph = "promedio_alquileres_por_hora"
franja = datos.cargar_tabla("P1_demanda_por_franja_horaria")
hora = datos.cargar_tabla("P1_demanda_por_hora_del_dia")
tipo = datos.cargar_tabla("P2_demanda_por_tipo_de_dia")
estacion = datos.cargar_tabla("P3_demanda_por_estacion")
p6 = datos.cargar_tabla("P6_variacion_2011_2012").set_index("tipo_usuario")
p5 = datos.cargar_tabla("P5_participacion_casuales_y_registrados")

c = dash.contenido()

def exacto(t):
    """Promedio por hora sin redondear (total / horas): las tablas de AE3 lo traen redondeado a 2 decimales."""
    return t["total_alquileres"] / t["horas_observadas"]

def en_texto(texto, valor, decimales=1):
    """True si el valor (con coma decimal y punto de miles) aparece escrito en el texto."""
    from analitica_bike.dashboard_bike import numero
    return numero(valor, decimales) in texto

# El texto redondea a 1 decimal desde el valor exacto (total / horas), no desde el redondeado a 2 decimales de AE3:
# 921.834 / 5.014 = 183,852 se escribe 183,9 (redondear dos veces daría 183,8).
fin_semana = tipo.loc[tipo["tipo_dia"] == "Fin de semana"].iloc[0]
assert fin_semana[ph] == 183.85 and en_texto(c["P2"]["respuesta"], fin_semana["total_alquileres"] / fin_semana["horas_observadas"])
assert "183,9" in c["P2"]["respuesta"] and "183,8" not in c["P2"]["respuesta"]

assert en_texto(c["P1"]["respuesta"], exacto(franja).max())
assert en_texto(c["P1"]["respuesta"], exacto(hora).max())
assert en_texto(c["P1"]["respuesta"], exacto(hora).min())
assert en_texto(c["P2"]["respuesta"], exacto(tipo).iloc[tipo["tipo_dia"].tolist().index("Laboral")])
assert en_texto(c["P3"]["respuesta"], exacto(estacion).max())
t5 = p5.loc[p5["dimension"] == "Total"].iloc[0]
assert en_texto(c["P5"]["respuesta"], t5["usuarios_casuales"] / t5["alquileres_total"] * 100)
T = p6.loc["Total"]
assert en_texto(c["P6"]["respuesta"], (T["total_2012"] - T["total_2011"]) / T["total_2011"] * 100)
assert en_texto(c["P6"]["respuesta"], T["total_2012"] / T["horas_observadas_2012"])
assert [r["id"] for r in dash.respuestas()] == ["P1", "P2", "P3", "P4", "P5", "P6"]
print("Los textos del dashboard usan las mismas cifras que las tablas de AE3 (redondeadas una sola vez, desde el valor exacto).")

Los textos del dashboard usan las mismas cifras que las tablas de AE3 (redondeadas una sola vez, desde el valor exacto).


## 4. Todas las figuras se construyen

Se construyen las figuras de Plotly de la app (sin filtros y con varios filtros, incluida una selección muy pequeña) y se comprueba que no fallan. Las figuras se ven en la app (`streamlit run AE4/app/app.py`); el notebook no guarda imágenes.

In [5]:
import plotly.graph_objects as go

def comprobar_figuras(d, etiqueta):
    figuras = d.todas_las_figuras()
    assert figuras and all(isinstance(f, go.Figure) for f in figuras.values())
    d.respuestas()
    print(f"{etiqueta}: {len(figuras)} figuras y 6 respuestas construidas ({d.horas:,} horas)".replace(",", "."))
    return figuras

figuras = comprobar_figuras(dash, "Sin filtros")
assert not dash.filtrar().filtros_activos()
comprobar_figuras(dash.filtrar(anios=[2012]), "Año 2012")
comprobar_figuras(dash.filtrar(estaciones=["Invierno"], tipos_dia=["Festivo"]), "Invierno y festivos")
comprobar_figuras(dash.filtrar(horas=(0, 3)), "Horas 0 a 3")
comprobar_figuras(dash.filtrar(tipos_dia=["Festivo"], horas=(8, 9)), "Festivos de 8 a 9 h")
print("\nFiguras de la app:", ", ".join(figuras))

# Muestra insuficiente (menos de 100 horas en total): ninguna pestaña recomienda ni rankea
chico = dash.filtrar(estaciones=["Invierno"], tipos_dia=["Festivo"], horas=(2, 3))
assert chico.insuficiente and chico.horas < 100
assert all("no se hacen recomendaciones" in v["decision"] and "no se hacen recomendaciones" in v["conclusion"]
           for v in chico.contenido().values())

# Rango de horas angosto pero con muestra suficiente por hora: la recomendación de P1 es válida
fin = dash.filtrar(tipos_dia=["Fin de semana"], horas=(7, 9))
assert not fin.insuficiente and fin.horas_por("hora").min() >= 100
assert "pocas horas del día" not in fin.contenido()["P1"]["decision"]
assert "Redistribución" in fin.contenido()["P1"]["decision"]
print("Muestra insuficiente y rango de horas angosto: comportamiento correcto.")

Sin filtros: 21 figuras y 6 respuestas construidas (17.379 horas)


Año 2012: 21 figuras y 6 respuestas construidas (8.734 horas)


Invierno y festivos: 21 figuras y 6 respuestas construidas (164 horas)


Horas 0 a 3: 21 figuras y 6 respuestas construidas (2.862 horas)


Festivos de 8 a 9 h: 21 figuras y 6 respuestas construidas (42 horas)

Figuras de la app: contexto_serie, contexto_histograma, P1_franja, P1_hora, P1_calor, P2_tipo_dia, P2_tipo_dia_franja, P2_perfil, P2_festivos, P3_estacion, P3_ranking, P3_linea, P4_clima, P4_temperatura, P5_Tipo de día, P5_Franja horaria, P5_Estación, P5_Tipo de festivo, P6_totales, P6_promedio, P6_mensual


Muestra insuficiente y rango de horas angosto: comportamiento correcto.


## 5. Cierre

In [6]:
# Reporte breve (el workflow de CI lee las líneas "Filas:" y "Columnas:")
RUTA_RESUMEN = Path("../resultados/resumen_ae4.txt")
RUTA_RESUMEN.parent.mkdir(parents=True, exist_ok=True)
RUTA_RESUMEN.write_text(
    f"Filas: {len(df)}\n"
    f"Columnas: {df.shape[1]}\n"
    f"Tablas verificadas contra AE3: {len(TABLAS)}\n"
    f"Figuras de la app construidas: {len(figuras)}\n",
    encoding="utf-8",
)
print(f"Reporte guardado: {RUTA_RESUMEN}")

Reporte guardado: ../resultados/resumen_ae4.txt
